In [463]:
import pandas as pd
import itertools
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from functools import reduce

In [464]:
def conv_to_float_arr(df):
    if isinstance(df['message_embedding'][0], str):
            df['message_embedding'] = [val[1:-1] for val in df['message_embedding']]
            df['message_embedding'] = [[float(e) for e in embedding.split(',')] for embedding in df['message_embedding']]
            df['message_embedding'] = [np.array(e) for e in df['message_embedding']]
    return df

In [465]:
#load the dataset
df = pd.read_csv('../output/jury_output_chat_level.csv')

#load the embeddings
sentence_vectors = pd.read_csv('../embeddings/jury_conversations_with_outcome_var.csv')

#merge the df and the embeddings
merged_df = pd.concat([df,sentence_vectors], axis=1)

conv_to_float_arr(merged_df)

,conversation_num,batch_num,round_num,speaker_hash,speaker_nickname,timestamp,message,majority_pct,num_flipped,flipped_pct,...,indirect_(greeting),direct_question,direct_start,haspositive,hasnegative,subjunctive,indicative,Unnamed: 0,message,message_embedding
0,0,0,0,5e7e1e0031f4e454e196c30b,niceRhino,2020-04-20T18:27:20.125Z,hello,1.000000,1,0.333333,...,1,0,0,0,0,0,0,0,Hello!,"[-0.06062667816877365, 0.03996481001377106, 0...."
1,0,0,0,5e31d6e4e31c5304c46f1413,culturedCow,2020-04-20T18:27:23.764Z,hi,1.000000,1,0.333333,...,1,0,0,0,0,0,0,1,Hi!,"[-0.08946911245584488, 0.03267906978726387, 0...."
2,0,0,0,5e7e4f4c31f4e454e196c9c4,spryBison,2020-04-20T18:27:27.724Z,hello,1.000000,1,0.333333,...,1,0,0,0,0,0,0,2,Hello,"[-0.06277179718017578, 0.05495883524417877, 0...."
3,0,0,0,5d482ea421c9be351f762255,youngLion,2020-04-20T18:27:30.410Z,hi,1.000000,1,0.333333,...,1,0,0,0,0,0,0,3,Hi,"[-0.09047617763280869, 0.04043959453701973, 0...."
4,0,0,0,5e84cc3c50f6e364321d6265,smallGiraffe,2020-04-20T18:27:35.506Z,hi,1.000000,1,0.333333,...,1,0,0,0,0,0,0,4,hi,"[-0.09047617763280869, 0.04043959453701973, 0...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14883,347,177,3,5f1f466846a7f03dda9b37c3,eagerElephant,2020-07-28T22:36:22.226Z,i think he is the asshole,0.666667,1,0.333333,...,0,0,0,0,0,0,0,14883,I think he is the asshole,"[-0.031704213470220566, 0.028208967298269272, ..."
14884,347,177,3,5f1a214246a7f03dda9a6aad,inspiredGiraffe,2020-07-28T22:36:46.155Z,he didnt vent to them or take up frustrations ...,0.666667,1,0.333333,...,0,0,0,0,1,0,0,14884,He didn't vent to them or take up frustrations...,"[0.07338573783636093, 0.07632912695407867, -0...."
14885,347,177,3,5f2096dab0aafc741f6865bb,inquisitiveBison,2020-07-28T22:36:49.238Z,very interested,0.666667,1,0.333333,...,0,0,0,0,0,0,0,14885,very interested,"[-0.034968696534633636, -0.04083353281021118, ..."
14886,347,177,3,5f1f472646a7f03dda9b3895,excitedDolphin,2020-07-28T22:36:51.487Z,im fine with venting on reddit i just dont thi...,0.666667,1,0.333333,...,0,0,0,1,1,0,0,14886,"I'm fine with venting on Reddit, I just don't ...","[0.060588538646698, 0.040215328335762024, -0.0..."


In [466]:
'''
Aggregates the embeddings at the speaker level
'''
def user_centroid(chat_data):
    # Get mean embedding per speaker per conversation
    return pd.DataFrame(chat_data.groupby(['conversation_num','speaker_nickname'])['message_embedding'].apply(np.mean))


'''
Calculates the lsm for every pair of users, and averages the pairwise embedddings to get the lsm for the conversation
'''

def language_style_matching(chat_data):
    #convert the embeddings into a numpy array
    array = np.array(chat_data['message_embedding'].tolist())

    #variable to hold the cosine similarity
    cosine = 0

    #total number of distinct pairs of speakers
    num_pairs = (len(array) * (len(array) -1)) / 2

    #iterate through the matrix and get the cosine similarity for each pair of speakers
    for i in range(0,len(array)):
        for j in range (0,len(array)):
            if i != j:
                speaker1_embeddings = array[i].reshape(1,-1)
                speaker2_embeddings = array[j].reshape(1,-1)
                pair_cosine = cosine_similarity(speaker1_embeddings,speaker2_embeddings)[0][0]
                cosine = cosine + pair_cosine
    
    #average cosine similarity. We additionally divide by 2 as each pair is counted twice
    average = cosine / (num_pairs * 2)

    chat_data['lsm'] = average


'''
Get the pairwise lsm for each conversation
'''

def lsm_pairwise_approach(df):

    pairwise_lsm = []

    #get the range of the conversation nums
    unique_values = df['conversation_num'].unique()
    value_range = unique_values.max() - unique_values.min() + 1

    #calculate average lsm for every conversation
    for i in range(0,value_range):
        convo = df[df['conversation_num'] == i]
        convo = user_centroid(convo)
        language_style_matching(convo)
        lsm = convo['lsm'].mean()
        pairwise_lsm.append(lsm)

    return pairwise_lsm

In [467]:
#Shrutis DD code modified to calculate LSM (no variable names have been changed)

def assign_chunk_nums(chat_data, num_chunks):
    list_conv_df = [tpl[1] for tpl in list(chat_data.groupby('conversation_num'))]
    split_conv_df = [assign_chunk_nums_helper(df, num_chunks) for df in list_conv_df]
    return reduce(lambda x, y: pd.concat([x, y], axis=0), split_conv_df)

def assign_chunk_nums_helper(chat_data, num_chunks):
    list_df = np.array_split(chat_data, num_chunks)
    for x in range(num_chunks):
        list_df[x]['chunk_num'] = str(x)
    return reduce(lambda x, y: pd.concat([x, y], axis=0), list_df)

#from given code 
def get_unique_pairwise_combos(lst):
    '''Computes all unique pairwise combinations of the elements in a list.
    input: array or list
    output: list of unique pairwise combinations of elements of the input list'''
    return list(itertools.combinations(lst, 2))


def get_cosine_similarity(vecs):
    '''computes cosine similarity between a list of vectors
    input: list of vectors (vecs) (this has to be a pair!)
    output: cosine similarity
    '''
    if len(vecs) > 1:
        cos_sim_matrix = cosine_similarity(vecs)
        return cos_sim_matrix[np.triu_indices(len(cos_sim_matrix), k = 1)]
    else:
        return np.nan
    
def get_variance_in_DD(chat_data):
    dd_results = chat_data.groupby(['conversation_num', 'chunk_num']).apply(get_DD)
    dd_results = dd_results.reset_index(drop=True)
    results = dd_results.groupby("conversation_num", as_index=False).var()[['conversation_num', 'lsm']]
    return results.rename(columns={'lsm_centroid': 'variance_in_lsm'})
    
def get_nan_vector():
    f = open("../data/vectors/nan_vector.txt", "r")
    str_vec = f.read()
    nan_vector_list = [float(e) for e in str_vec[1:-1].split(',')]
    return np.array(nan_vector_list)


def get_within_person_disc_range(chat_data, num_chunks):

    # Get nan vector 
    nan_vector = get_nan_vector()

    #calculate mean vector per speaker per chunk
    mean_vec_speaker_chunks = pd.DataFrame(chat_data.groupby(['conversation_num', 'speaker_nickname', 'chunk_num']).message_embedding.apply(np.mean)).unstack('chunk_num').rename(columns={'message_embedding': 'mean_chunk_vec'})

    #collapse multi-index
    mean_vec_speaker_chunks.columns = ["_c".join(col).strip() for col in mean_vec_speaker_chunks.columns.values]

    actual_num_chunks = len(mean_vec_speaker_chunks[2:].columns) # omit the first two, which is conversation_num and speaker_nickname

    #each element in inter_chunk_range is a list of cosine distances BETWEEN each pair of consecutive chunks
    inter_chunk_range = [ [] for i in range(actual_num_chunks - 1)]

    for conv_idx, row in mean_vec_speaker_chunks.iterrows():
        for index in range(actual_num_chunks - 1):
            tpl = [row['mean_chunk_vec_c' + str(index)], row['mean_chunk_vec_c'+ str(index+1)]]
            value = 0
            if (pd.isnull(tpl)).all():
                value = np.nan
            elif (pd.isnull(tpl)).any():
                # Compare with Nanvector 
                if type(tpl[0]) == float:     
                    tpl = [nan_vector, tpl[1]]
                else:
                    tpl = [nan_vector, tpl[0]]
                value = get_cosine_similarity(tpl)[0]
            else:
                value = get_cosine_similarity(tpl)[0]
            inter_chunk_range[index].append(value)

    index = []
    for i in range(actual_num_chunks - 1):
        index.append("c" + str(i) + "_c" + str(i + 1))
    range_df = pd.DataFrame(inter_chunk_range, index=index).T
    range_df['conversation_num'] = mean_vec_speaker_chunks.reset_index()['conversation_num']
    range_df = range_df.set_index('conversation_num')

    # variance within person discursive range 
    var_disc_range = range_df.groupby('conversation_num').apply(lambda x: np.nanvar(x, axis=0).sum()).to_frame().rename(columns={0:'incongruent_modulation'})
    
    # average within person discursive range 
    avg_disc_range = range_df.groupby('conversation_num').apply(lambda x: np.nanmean(x, axis=0).sum()).to_frame().rename(columns={0:'within_person_disc_range'})

    return pd.merge(
                left=var_disc_range,
                right=avg_disc_range,
                on=['conversation_num'],
                how="inner"
            )
def get_DD(chat_data):
    
    # Get mean embedding per speaker per conversation
    user_centroid_per_conv = pd.DataFrame(chat_data.groupby(['conversation_num','speaker_nickname'])['message_embedding'].apply(np.mean)).reset_index().rename(columns={'message_embedding':'mean_embedding'})

    # For each team(conversation) get all unique pairwise combinations of members' means:
    user_pairs = pd.DataFrame(user_centroid_per_conv.groupby(['conversation_num'])['mean_embedding'].\
    apply(get_unique_pairwise_combos)).reset_index().\
    rename(columns={'mean_embedding':'user_pairs_per_conv'})

    # Get cosine distances between each pair for every conversation, average all the distances to get DD per conversation
    cos_dists_mean_widay_btwu = []

    for lst in user_pairs.user_pairs_per_conv:

        # Make sure list isn't empty:
        if lst:
            # Store the cosine distances for the person's list of tuples
            cos_dists = []
            for tpl in lst:
                try:
                    cos_d = get_cosine_similarity([tpl[0], tpl[1]])
                    cos_dists.append(cos_d)
                except ValueError as e:
                    # Occurs when np.nan in tuple
                    pass

            # Compute mean of cos dists
            cos_dists_mean_widay_btwu.append(np.nanmean(cos_dists, dtype="float64"))
        else:
            cos_dists_mean_widay_btwu.append(np.nan)

    user_pairs['lsm'] =  cos_dists_mean_widay_btwu

    ## split into chunks 

    return user_pairs[['conversation_num', 'lsm_centroid']]


def lsm_centroid_approach(chat_data,vect_data):

    chats = chat_data.copy()

    # Format data
    chats['message_embedding'] = conv_to_float_arr(vect_data['message_embedding'].to_frame())

    # Get discursive diversity
    disc_div = get_DD(chats)
    disc_div = disc_div.replace(np.nan, 0)

    num_chunks = 1 # TODO - this is where we will more intelligently assign chunks; currently chose 3 based on EDA

    # Split into chunks 
    chats_chunked = assign_chunk_nums(chats, num_chunks)

    # Get variance in discursive diversity 
    var_disc_div = get_variance_in_DD(chats_chunked)
    var_disc_div = var_disc_div.replace(np.nan, 0)

    # Get within-person discursive range metrics
    modulation_metrics = get_within_person_disc_range(chats_chunked, num_chunks)
    modulation_metrics = modulation_metrics.replace(np.nan, 0)

    dd_features = [disc_div, var_disc_div, modulation_metrics]
    return reduce(lambda x, y: pd.merge(x, y, on = 'conversation_num'), dd_features)

In [468]:
# Priya's Average Based Approach
pairwise_result = lsm_pairwise_approach(merged_df)

In [469]:
# Shruti's Centroid Based Approach
df = lsm_centroid_approach(df,sentence_vectors)
print(df)

     conversation_num       lsm  variance_in_lsm  incongruent_modulation  \
0                   0  0.593345              0.0                     0.0   
1                   1  0.637275              0.0                     0.0   
2                   2  0.504474              0.0                     0.0   
3                   3  0.583008              0.0                     0.0   
4                   4  0.669423              0.0                     0.0   
..                ...       ...              ...                     ...   
343               343  0.441943              0.0                     0.0   
344               344  0.521968              0.0                     0.0   
345               345  0.546626              0.0                     0.0   
346               346  0.556654              0.0                     0.0   
347               347  0.566788              0.0                     0.0   

     within_person_disc_range  
0                         0.0  
1                      

In [470]:
#TESTING

#drop some columns
columns_to_drop = ['variance_in_lsm','incongruent_modulation','within_person_disc_range']
df = df.drop(columns=columns_to_drop)

#add the pairwise result to compare
df['lsm_pairwise'] = pd.Series(pairwise_result) 

#get the difference between the centroid based approach and the pairwise approach
df['Difference'] = round(df['lsm'] - df['lsm_pairwise'],5)

df.to_csv('lsm_testing.csv')